# Compare mean AUC(t) across models: time-to-platinum, landmarks 0 / 90 / 180

Reads existing outputs only (nothing is refit):

- **03_multivariate** -- elastic-net Cox and XGBoost (`both` = clinical+labs, `baseline`)
- **03b_multivariate_longitudinal** -- Dynamic-DeepHit (landmark arm, plus dynamic arm / SurvLatent ODE if run)
- **predictive_checklists** -- checklist and its baselines

**Caveat.** The first two families report `test_mean_auc_t`, the time-dependent AUC averaged over the
evaluation grid. Predictive checklists are fit per fixed horizon (1-5 y) and report a single-horizon AUC, so they
have no `mean_auc_t`. The checklist panel uses the *unweighted mean of its per-horizon AUCs*, which is a related but
**not identical** quantity. It is plotted separately and hatched, and the per-horizon AUCs are shown too.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
import compass_pipeline as cp

# Match the settings used in 03 / 03b so the same output tree and task list resolve.
cp.RUN_DYNAMIC = False        # set True if 03b ran the per-timepoint Dynamic-DeepHit arm
cp.RUN_SURVLATENT = False     # set True if 03b ran SurvLatent ODE
cp.LONGITUDINAL_INCLUDE_COMPETING = False  # keep the cause-only config (comparable to Cox/XGBoost)

ENDPOINT = "platinum"
LANDMARKS = [0, 90, 180]
RUNS = cp.make_endpoint_runs(["adt"], endpoints=(ENDPOINT,), cohorts=("all",), exclusions=("none",))

CHECKLIST_DIR = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis/predictive_checklists/full_adt")
CHECKLIST_POPULATION = None   # None = every population in metrics.csv; or e.g. "all"


In [ ]:
# Cox / XGBoost (03) and Dynamic-DeepHit etc. (03b): canonical schema, so one concat serves both.
frames = []
for run in RUNS:
    frames.append(cp.summarize_outputs(run))
    frames.append(cp.summarize_longitudinal_outputs(run))
summary = pd.concat(frames, ignore_index=True)
summary = summary[summary["landmark"].isin(LANDMARKS)]
summary["label"] = summary["model"] + " (" + summary["config"] + ")"
missing = summary[summary["status"] != "ok"]
if len(missing):
    print("Not available:"); display(missing[["model", "config", "landmark", "status"]])
mean_auc = summary.pivot_table(index="label", columns="landmark", values="mean_auc_t", sort=False)
mean_auc.round(3)

In [ ]:
# Predictive checklists: per-horizon AUC -> mean over horizons (NOT the same estimand as mean_auc_t).
ck_path = CHECKLIST_DIR / "metrics.csv"
if ck_path.exists():
    ck = pd.read_csv(ck_path)
    ck = ck[(ck["endpoint"] == ENDPOINT) & (ck["status"] == "ok") & ck["landmark_days"].isin(LANDMARKS)]
    if CHECKLIST_POPULATION is not None:
        ck = ck[ck["population"] == CHECKLIST_POPULATION]
    print("populations:", sorted(ck["population"].unique()), "| models:", sorted(ck["model"].unique()))
    ck_h = ck.pivot_table(index=["population", "model", "landmark_days"], columns="horizon_days", values="auc")
    ck_mean = ck_h.mean(axis=1).rename("mean_auc_over_horizons").reset_index()
    display(ck_h.round(3))
else:
    ck_h, ck_mean = None, None
    print(f"No checklist output at {ck_path}")

In [ ]:
fig, axes = plt.subplots(1, len(LANDMARKS), figsize=(5 * len(LANDMARKS), 4.5), sharey=True)
for ax, lm in zip(np.atleast_1d(axes), LANDMARKS):
    sub = mean_auc[lm].dropna() if lm in mean_auc else pd.Series(dtype=float)
    bars = list(sub.items())
    names = [n for n, _ in bars]; vals = [v for _, v in bars]
    hatch = [False] * len(bars)
    if ck_mean is not None:
        c = ck_mean[ck_mean["landmark_days"] == lm]
        if CHECKLIST_POPULATION is None and c["population"].nunique() > 1:
            c = c[c["population"] == sorted(c["population"].unique())[0]]
        for _, r in c.iterrows():
            names.append(f"{r['model']} [horizon-mean]"); vals.append(r["mean_auc_over_horizons"]); hatch.append(True)
    order = np.argsort(vals)
    y = np.arange(len(vals))
    for i, j in enumerate(order):
        ax.barh(i, vals[j], hatch="//" if hatch[j] else None, color="0.6" if hatch[j] else "C0", edgecolor="white")
        ax.text(vals[j] + 0.005, i, f"{vals[j]:.3f}", va="center", fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels([names[j] for j in order], fontsize=8)
    ax.axvline(0.5, color="k", lw=0.6, ls=":")
    ax.set_xlim(0.4, 1.0); ax.set_title(f"Landmark {lm} d"); ax.set_xlabel("mean AUC(t)")
fig.suptitle("Time-to-platinum: mean AUC(t) by model (hatched = checklist mean over 1-5 y horizons)")
fig.tight_layout(); plt.show()

In [ ]:
# Per-horizon checklist AUC alongside the comparison, to see what the horizon-mean hides.
if ck_h is not None and len(ck_h):
    fig, axes = plt.subplots(1, len(LANDMARKS), figsize=(5 * len(LANDMARKS), 3.8), sharey=True)
    for ax, lm in zip(np.atleast_1d(axes), LANDMARKS):
        if lm in ck_h.index.get_level_values("landmark_days"):
            for (pop, model), row in ck_h.xs(lm, level="landmark_days").iterrows():
                ax.plot(row.index / 365.25, row.values, marker="o", label=f"{model} ({pop})")
        ax.set_title(f"Landmark {lm} d"); ax.set_xlabel("horizon (years)"); ax.axhline(0.5, color="k", lw=0.6, ls=":")
    np.atleast_1d(axes)[0].set_ylabel("AUC at horizon"); np.atleast_1d(axes)[-1].legend(fontsize=7)
    fig.tight_layout(); plt.show()

## What each model uses at the 180-day landmark

- **Elastic-net Cox**: non-zero coefficients (standardized covariates, so `coef` is comparable across features).
- **XGBoost**: gain importance.
- **Predictive checklist**: the checklist itself, per horizon.
- **Dynamic-DeepHit**: neural network, no feature-level output is written, so it is skipped.

Coefficient / importance CSVs are located by glob and identified by their columns, because the pipeline's per-landmark
file naming differs between layouts. Anything not found is reported rather than raising.

In [ ]:
FEATURE_LANDMARK = 180
TOP_N = 15
run = RUNS[0]

def find_csv(model_dir, config_dir, include, exclude=(), need_cols=()):
    base = run["output_dir"] / cp.model_output_dir(model_dir) / f"landmark_{FEATURE_LANDMARK}" / config_dir
    for p in sorted(base.glob("*.csv")):
        n = p.name
        if include in n and not any(x in n for x in exclude):
            cols = pd.read_csv(p, nrows=0).columns
            if all(c in cols for c in need_cols):
                return p
    print(f"[not found] {model_dir}/{config_dir}: *{include}* under {base}")

def top_bar(df, col, title, n=TOP_N, signed=True):
    d = df.reindex(df[col].abs().sort_values(ascending=False).index).head(n)[::-1]
    fig, ax = plt.subplots(figsize=(7, 0.32 * len(d) + 1))
    ax.barh(d["feature"], d[col], color=["C3" if v > 0 else "C0" for v in d[col]] if signed else "C0")
    ax.set_title(title); ax.set_xlabel(col); fig.tight_layout(); plt.show()

for cfg in ("both", "baseline"):
    prefix = "cox_agg_multivariable" if cfg == "both" else "cox_agg_baseline"
    p = find_csv("elastic-net", cfg, prefix, exclude=("metrics", "auc_t", "brier", "risks", "selection", "folds", "grid"),
                 need_cols=("feature", "coef"))
    if p is not None:
        d = pd.read_csv(p)
        d = d[(d["endpoint"] == ENDPOINT) & (d["coef"] != 0)] if "endpoint" in d else d[d["coef"] != 0]
        print(f"Elastic-net Cox [{cfg}]: {len(d)} non-zero coefficients ({p.name})")
        top_bar(d, "coef", f"Elastic-net Cox [{cfg}], landmark {FEATURE_LANDMARK} d: top |coef| (red = higher risk)")

for cfg in ("both", "baseline"):
    p = find_csv("xgboost", cfg, "feature_importance", need_cols=("feature", "gain"))
    if p is not None:
        d = pd.read_csv(p)
        if "endpoint" in d:
            d = d[d["endpoint"] == ENDPOINT]
        print(f"XGBoost [{cfg}] ({p.name})")
        top_bar(d, "gain", f"XGBoost [{cfg}], landmark {FEATURE_LANDMARK} d: top gain", signed=False)

In [ ]:
# Predictive checklists at the chosen landmark: the checklist.txt written for each horizon.
import json
found = False
for cl in sorted(CHECKLIST_DIR.glob(f"**/horizon_*/checklist.json")):
    parts = cl.relative_to(CHECKLIST_DIR).parts
    ident = json.loads((cl.parent / "model.json").read_text())["identity"]
    if ident.get("endpoint") != ENDPOINT or ident.get("landmark_days") != FEATURE_LANDMARK:
        continue
    if CHECKLIST_POPULATION is not None and ident.get("population") != CHECKLIST_POPULATION:
        continue
    found = True
    print("=" * 78); print(f"population={ident.get('population')}  horizon={ident.get('horizon_days')} d"); print("=" * 78)
    print((cl.parent / "checklist.txt").read_text())
if not found:
    print(f"No {ENDPOINT} checklists for landmark {FEATURE_LANDMARK} under {CHECKLIST_DIR}")

## C-index, and a like-for-like comparison with the checklists

**C-index.** Cox / XGBoost / Dynamic-DeepHit report `test_c_index` (already in `summary`). Checklists don't, but each
horizon's `predictions.csv` holds a test-set `score` per patient, so Harrell's C is computed here by joining those scores to the
landmark table's `t_platinum` / `PLATINUM`. A checklist score is an integer item count, so many ties (tied risk pairs count 0.5)
and one score per horizon: the c-index is reported per horizon, not pooled.

**Direct comparison.** The comparable quantity is the *horizon-specific* AUC. The survival models write
`test_auc_h<N>` (N in weeks; 1 y is about h52) from IPCW time-dependent AUC; the checklist reports a
censoring-weighted AUC for 'event within H days'. Both are cases-by-horizon vs controls with censoring handled by inverse
weighting, so at matching horizons they are comparable, though not the same estimator. The test sets are the same patient split.

In [ ]:
import re, json
from sksurv.metrics import concordance_index_censored

sys.path.insert(0, str(Path.cwd().parents[1]))
from COMPASS.survival_analysis.predictive_checklists.data import load_landmark

# ---- survival models: c-index (from summary) -------------------------------------------
cindex = summary.pivot_table(index="label", columns="landmark", values="c_index", sort=False)
display(cindex.round(3))

# ---- checklists: per-horizon test c-index from predictions.csv ------------------------
def checklist_cells(landmark):
    for model_json in sorted(CHECKLIST_DIR.glob("**/horizon_*/model.json")):
        ident = json.loads(model_json.read_text())["identity"]
        if ident.get("endpoint") == ENDPOINT and ident.get("landmark_days") == landmark:
            if CHECKLIST_POPULATION is None or ident.get("population") == CHECKLIST_POPULATION:
                yield model_json.parent, ident

ck_rows = []
for lm in LANDMARKS:
    frame = load_landmark(RUNS[0]["inputs_dir"], ENDPOINT, lm).frame.set_index("DFCI_MRN")
    for folder, ident in checklist_cells(lm):
        for pred_path in [folder / "predictions.csv", *sorted(folder.glob("predictions_*.csv"))]:
            pred = pd.read_csv(pred_path)
            pred = pred[pred["dataset"] == "test"].join(frame[[f"t_{ENDPOINT}", ENDPOINT.upper()]], on="DFCI_MRN").dropna()
            name = "checklist" if pred_path.name == "predictions.csv" else pred_path.stem.removeprefix("predictions_")
            if pred["score"].nunique() < 2 or pred[ENDPOINT.upper()].sum() == 0:
                continue
            c = concordance_index_censored(pred[ENDPOINT.upper()].astype(bool), pred[f"t_{ENDPOINT}"], pred["score"])[0]
            ck_rows.append(dict(landmark=lm, population=ident["population"], horizon_days=ident["horizon_days"], model=name, c_index=c))
ck_c = pd.DataFrame(ck_rows)
if len(ck_c):
    display(ck_c.pivot_table(index=["population", "model", "landmark"], columns="horizon_days", values="c_index").round(3))

In [ ]:
# ---- like-for-like: horizon-specific AUC, survival models vs checklists ------------------
# Survival-model metrics files hold test_auc_h<N> (N in weeks); match to checklist horizons within one week.
def survival_horizon_auc():
    rows = []
    for run in RUNS:
        for model, lm, cfg, fname in (cp.tasks_for_run(run, cp.MULTIVARIATE_TASK_SPECS)
                                       + cp.tasks_for_run(run, cp.longitudinal_task_specs(ENDPOINT))):
            if lm not in LANDMARKS:
                continue
            fname = cp.longitudinal_metrics_filename(model, cfg, lm, fname)
            p = run["output_dir"] / cp.model_output_dir(model) / f"landmark_{lm}" / cfg / fname
            if not p.exists():
                continue
            d = pd.read_csv(p)
            d = d[d["endpoint"] == ENDPOINT] if "endpoint" in d else d
            if d.empty:
                continue
            for col in d.columns:
                m = re.fullmatch(r"test_auc_h(\d+)", col)
                if m and pd.notna(d.iloc[0][col]):
                    rows.append(dict(model=f"{model} ({cfg})", landmark=lm, horizon_days=int(m[1]) * 7, auc=float(d.iloc[0][col])))
    return pd.DataFrame(rows)

sv = survival_horizon_auc()
if sv.empty:
    print("No test_auc_h* columns found in the survival-model metrics files.")
if ck is not None and len(ck):
    cl = ck.rename(columns={"landmark_days": "landmark"})[["model", "landmark", "horizon_days", "auc"]]
    cl["model"] = "checklist: " + cl["model"]
    # snap each survival horizon to the nearest checklist horizon within 7 days
    grid = np.array(sorted(cl["horizon_days"].unique()))
    if len(sv):
        sv["nearest"] = [grid[np.abs(grid - h).argmin()] for h in sv["horizon_days"]]
        sv = sv[np.abs(sv["nearest"] - sv["horizon_days"]) <= 7].assign(horizon_days=lambda x: x["nearest"]).drop(columns="nearest")
        if sv.empty:
            print("Survival-model AUC horizons do not fall within 7 days of any checklist horizon "
                  f"({list(grid)} d); re-run the survival models with matching AUC horizons for a direct comparison.")
    both = pd.concat([sv, cl], ignore_index=True)
    display(both.pivot_table(index=["landmark", "model"], columns="horizon_days", values="auc", sort=False).round(3))

    fig, axes = plt.subplots(1, len(LANDMARKS), figsize=(5 * len(LANDMARKS), 4), sharey=True)
    for ax, lm in zip(np.atleast_1d(axes), LANDMARKS):
        for model, g in both[both["landmark"] == lm].groupby("model", sort=False):
            g = g.groupby("horizon_days")["auc"].mean().sort_index()
            ax.plot(g.index / 365.25, g.values, marker="o", ls="--" if model.startswith("checklist") else "-", label=model)
        ax.axhline(0.5, color="k", lw=0.6, ls=":"); ax.set_title(f"Landmark {lm} d"); ax.set_xlabel("horizon (years)")
    np.atleast_1d(axes)[0].set_ylabel("AUC at horizon"); np.atleast_1d(axes)[-1].legend(fontsize=6)
    fig.suptitle("Horizon-specific AUC, full survival models vs predictive checklists (dashed)"); fig.tight_layout(); plt.show()